[![pythonista.io](img/pythonista.png)](https://www.pythonista.io)

# SBOM, provenance y attestations.

Este bloque parte de la SBOM de runners y la extiende hacia la trazabilidad del artefacto producido por nuestro propio pipeline.

## Objetivos.

- Entender para que sirve una SBOM.
- Relacionar build reproducible con provenance.
- Preparar politicas de verificacion antes del despliegue.

## Por que importa la trazabilidad.

Una SBOM describe que componentes participaron en la construccion de un artefacto. GitHub publica SBOM de sus runners; en Py271 damos el siguiente paso: necesitamos saber que dependencias, base image, herramientas y procesos intervinieron en la imagen o paquete que despues sera desplegado.

Sin trazabilidad, una imagen puede existir; con trazabilidad, una imagen puede justificarse.

## Attestations: SBOM firmada y verificable.

Una attestation combina dos conceptos: el contenido de la SBOM y una firma criptografica que acredita quien la genero y cuando. El resultado es un documento que no solo describe los componentes del artefacto, sino que ademas puede ser verificado de forma independiente.

Cosign puede adjuntar una attestation directamente al registro de imagenes, asociada al digest de la imagen:

```bash
# adjuntar SBOM como attestation a la imagen
cosign attest --yes \
  --predicate sbom.spdx.json \
  --type spdxjson \
  ghcr.io/mi-org/mi-app@sha256:abc123...

# verificar que la attestation existe y es valida
cosign verify-attestation \
  --type spdxjson \
  --certificate-identity-regexp "https://github.com/mi-org/.*" \
  --certificate-oidc-issuer "https://token.actions.githubusercontent.com" \
  ghcr.io/mi-org/mi-app@sha256:abc123...
```

La verificacion confirma tres cosas: que la attestation fue generada por un workflow de GitHub Actions, que el certificado esta dentro del periodo de validez registrado en Rekor, y que el contenido no fue modificado despues de la firma.

Esto permite implementar politicas de despliegue que rechacen imagenes sin attestation verificable, cerrando el ciclo entre construccion y operacion.

## Pipeline completo: SBOM, firma y attestation.

El flujo que une Syft, Cosign y GitHub Actions en un solo job:

```yaml
jobs:
  sbom-sign:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      packages: write
      id-token: write       # requerido para firma keyless con OIDC

    steps:
      - uses: actions/checkout@v4

      - name: Instalar Syft
        uses: anchore/sbom-action/download-syft@v0.15.0

      - name: Instalar Cosign
        uses: sigstore/cosign-installer@v3.5.0

      - name: Construir imagen
        run: docker build -t ghcr.io/${{ github.repository }}:${{ github.sha }} .

      - name: Push imagen
        run: docker push ghcr.io/${{ github.repository }}:${{ github.sha }}

      - name: Generar SBOM de la imagen
        run: |
          syft ghcr.io/${{ github.repository }}:${{ github.sha }} \
            -o spdx-json=sbom.spdx.json

      - name: Firmar imagen (keyless)
        run: |
          cosign sign --yes \
            ghcr.io/${{ github.repository }}:${{ github.sha }}

      - name: Adjuntar SBOM como attestation
        run: |
          cosign attest --yes \
            --predicate sbom.spdx.json \
            --type spdxjson \
            ghcr.io/${{ github.repository }}:${{ github.sha }}

      - name: Subir SBOM como artefacto del workflow
        uses: actions/upload-artifact@v4
        with:
          name: sbom
          path: sbom.spdx.json
```

El permiso `id-token: write` es el que habilita a Cosign para obtener un token OIDC de GitHub y firmar sin llaves estaticas. Sin ese permiso, la firma keyless falla.

In [ ]:
sbom = ['python==3.11', 'flask==3.0.0', 'gunicorn==22.0.0']

for dependencia in sbom:
    print('componente:', dependencia)

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>